based on cca_simulation2, I want to see if using cross validation i can obtain better weight estimates (more similar ot the gt weights) using less trials.

- My hyperparameter is the L2 regularization term
- My parameters are the time lag and the weights

i am using ridge regression and minimizing the mse to select lambda, but also the lag and weights

In [1]:
from cca_simulation2_script import *

In [ ]:
# ==== add below your helpers in cca_simulation2_script.py =====================

from dataclasses import dataclass
from typing import List, Tuple, Dict
import numpy as np
import pandas as pd
from cca_zoo.linear import rCCA
import matplotlib.pyplot as plt
  
def softmax(w: np.ndarray, tau: float = 1.0) -> np.ndarray:
    z = (w / max(tau, 1e-12)).astype(float)
    z -= z.max()                   # numerical stability
    e = np.exp(z)
    return e / (e.sum() + 1e-12)

def ridge_wx(X: np.ndarray, y: np.ndarray, lam: float, eps: float = 1e-12, normalise: bool = False ) -> np.ndarray:
    """
    Return ridge-regression weights w for y ≈ Xw.

    Same covariance-style formulation as before, but:
    * no softmax
    * optional L2 normalisation if normalise=True (default False).
    """
    # centre like np.cov does
    Xc = X - X.mean(axis=0, keepdims=True)
    yc = y - y.mean()
    n  = Xc.shape[0]
    Sxx = (Xc.T @ Xc) / max(n - 1, 1)      # ddof=1 covariance
    Sxy = (Xc.T @ yc) / max(n - 1, 1)

    A = Sxx + lam * np.eye(Sxx.shape[0])
    w = np.linalg.solve(A, Sxy)

    return w
    

def corr_with_weights(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    u = X @ w
    # y is already z-scored per trial in concat_trials; u doesn't need scaling for corr
    if u.std(ddof=0) == 0 or y.std(ddof=0) == 0:
        return np.nan
    return float(np.corrcoef(u, y)[0, 1])

def mse_with_weights(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    """
    Mean squared error between y and Xw.
    Assumes X, y already preprocessed by concat_trials.
    """
    y_hat = X @ w
    return float(np.mean((y_hat - y) ** 2))


# ---------------- per-(lag, λ) training step ---------------------------------
def fit_at_shift_lambda(trials: List[dict], shift_samples: int, lam: float, trim_ms: int, normalise: bool = True) -> Tuple[np.ndarray, float]:
    """
    Fit weights on concatenated TRAIN trials aligned with the given shift.
    Returns (w, r_train).
    """
    X_tr, y_tr = concat_trials(trials, shift_samples=shift_samples, trim_ms=trim_ms)

    if len(y_tr) == 0:
        return np.zeros(trials[0]['X'].shape[1]), np.nan
    #w_zoo = ridge_cca_wx_zoo(X_tr, y_tr, lam)

    w_ridge = ridge_wx(X_tr, y_tr, lam, normalise=False)
    train_mse = mse_with_weights(X_tr, y_tr, w_ridge)

    r_train = corr_with_weights(X_tr, y_tr, w_ridge)
    return w_ridge, train_mse, r_train

def evaluate_on_trials(trials: List[dict], shift_samples: int, w: np.ndarray, trim_ms: int) -> float:
    """Correlation on a (TRAIN or TEST) split using fixed shift and weights."""
    X_te, y_te = concat_trials(trials, shift_samples=shift_samples, trim_ms=trim_ms)
    if len(y_te) == 0:
        return np.nan
    #return corr_with_weights(X_te, y_te, w)
    return mse_with_weights(X_te, y_te, w), corr_with_weights(X_te, y_te, w)

def triplet_splits_from_trials(trials: List[dict], seed: int = 7) -> List[Tuple[np.ndarray, np.ndarray]]:
    """
    Leave-one-triplet-out using the 'triplet_id' present in each trial.
    Returns a list of (train_idx, test_idx) pairs; each test_idx has 3 indices.
    """
    import numpy as np
    rng = np.random.default_rng(seed)

    # map triplet_id -> list of trial indices
    id_to_idx = {}
    for i, tr in enumerate(trials):
        tid = tr.get('triplet_id')
        if tid is None:
            raise ValueError("Trial missing 'triplet_id'; ensure simulator sets it.")
        id_to_idx.setdefault(tid, []).append(i)

    trip_ids = list(id_to_idx.keys())
    rng.shuffle(trip_ids)

    splits = []
    all_idx = np.arange(len(trials))
    for tid in trip_ids:
        test_idx = np.array(sorted(id_to_idx[tid]), dtype=int)  # exactly 3
        train_idx = np.setdiff1d(all_idx, test_idx, assume_unique=False)
        splits.append((train_idx, test_idx))
    return splits



# ---------------- CV harness --------------------------------------------------
@dataclass
class CVResult:
    lam: float
    mean_train_r: float
    mean_test_r: float
    mean_train_mse: float
    mean_test_mse: float
    per_fold: List[Dict]            # diagnostics

def kfold_indices(n_trials: int, k: int, rng: np.random.Generator) -> List[Tuple[np.ndarray, np.ndarray]]:
    idx = np.arange(n_trials)
    rng.shuffle(idx)
    folds = np.array_split(idx, k)
    splits = []
    for f in range(k):
        test_idx = folds[f]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != f])
        splits.append((train_idx, test_idx))
    return splits

from typing import Optional

def cross_validate_l2_and_lag(
    trials: List[dict],
    lambdas: List[float],
    candidate_shifts: np.ndarray,
    trim_ms: int = 200,          # safety edge-trim (10 ms samples → 20 samples)
    k_folds: int = 4,
    seed: int = 7,
    splits: Optional[List[Tuple[np.ndarray, np.ndarray]]] = None,  # may be LOTO triplets
    print_splits: bool = False,   # NEW: print split diagnostics
    normalise: bool = True
) -> Tuple[List[CVResult], Dict]:
    """
    If 'splits' is provided, use those (train_idx, test_idx) pairs directly.
    Otherwise fall back to k-fold splits created from k_folds & seed.

    Returns:
      ordered_results: list[CVResult] sorted by mean_test_mse (asc, tie→lower λ)
      final_fit: {'lambda','best_shift','w','train_mse_all','train_rs_all',
                  'cv_mean_test_mse','cv_mean_test_r',
                  'cv_mean_train_mse','cv_mean_train_r','cv_details'}
    """
    rng = np.random.default_rng(seed)
    if splits is None:
        splits = kfold_indices(len(trials), k_folds, rng)  # old behavior

    # ---- Split diagnostics ----
    if print_splits:
        print(f"[CV] Using {len(splits)} folds")
        for f, (_, test_idx) in enumerate(splits, start=1):
            test_idx = np.array(test_idx, dtype=int)
            trip_ids = [trials[i].get('triplet_id', None) for i in test_idx]
            lens     = [trials[i].get('length_s', None)   for i in test_idx]
            uniq_trip = set([t for t in trip_ids if t is not None])
            msg_trip  = f"triplet_id(s)={sorted(list(uniq_trip))}" if any(t is not None for t in trip_ids) else "triplet_id(s)=<n/a>"
            msg_len   = f"length_s={lens}" if any(L is not None for L in lens) else "length_s=<n/a>"
            print(f"  - Fold {f:02d}: test_idx={test_idx.tolist()} | {msg_trip} | {msg_len} | n_test={len(test_idx)}")
            # quick sanity checks (non-fatal)
            if len(test_idx) == 3 and len(uniq_trip) == 1:
                print("    ✓ looks like a single triplet (3 trials, same triplet_id)")
            elif len(test_idx) == 3:
                print("    ! 3 trials but multiple/unknown triplet_id; check simulator tags.")
            else:
                print("    ! test size != 3; this is fine for standard k-fold but not LOTO-triplet.")

    # ---- CV across lambdas -------------------------------------------
    cv_summaries: List[CVResult] = []

    for lam in lambdas:
        print(f"[CV] Evaluating λ factor {lam:.6g} out of {len(lambdas)}")
        fold_rows = []
        train_rss, test_rss = [], []
        train_mses, test_mses = [], []

        for train_idx, test_idx in splits:
            train_trials = [trials[i] for i in train_idx]
            test_trials  = [trials[i] for i in test_idx]

            # 1) search best lag on TRAIN for this λ (MINIMISE train MSE)
            best_shift, best_train_mse, rs_best, best_w = None, np.inf, np.nan, None
            for s in candidate_shifts:
                w_s, mse_tr, rs_tr = fit_at_shift_lambda(
                    train_trials, int(s), lam, trim_ms, normalise=normalise
                )
                if np.isfinite(mse_tr) and mse_tr < best_train_mse:
                    best_train_mse, best_shift, rs_best, best_w = mse_tr, int(s), rs_tr, w_s

            # 2) evaluate on TEST using best (lag, w)
            mse_te, rs_te = evaluate_on_trials(test_trials, best_shift, best_w, trim_ms)

            fold_rows.append({
                "lam": lam, "shift": best_shift,
                "train_mse": best_train_mse, "test_mse": mse_te,
                "r_train": rs_best, "r_test": rs_te,
                "n_train": len(train_idx), "n_test": len(test_idx),
            })
            train_rss.append(rs_best);  test_rss.append(rs_te)
            train_mses.append(best_train_mse); test_mses.append(mse_te)

        cv_summaries.append(CVResult(
            lam=lam,
            mean_train_r=float(np.nanmean(train_rss)),
            mean_test_r=float(np.nanmean(test_rss)),
            mean_train_mse=float(np.nanmean(train_mses)),
            mean_test_mse=float(np.nanmean(test_mses)),
            per_fold=fold_rows,
        ))

    # pick λ with LOWEST mean TEST MSE (tie-breaker: smaller λ)
    ordered_results = sorted(cv_summaries, key=lambda c: (c.mean_test_mse, c.lam))
    best = ordered_results[0]

    # ------- recompute DEFINITIVE lag + weights at the chosen λ on ALL trials
    best_shift_all, best_mse_all, best_rs_all, best_w_all = None, np.inf, np.nan, None
    for s in candidate_shifts:
        w_s, mse_all, rs_all = fit_at_shift_lambda(
            trials, int(s), best.lam, trim_ms, normalise=normalise
        )
        if np.isfinite(mse_all) and mse_all < best_mse_all:
            best_mse_all, best_shift_all, best_rs_all, best_w_all = mse_all, int(s), rs_all, w_s

    final_fit = {
        "lambda": best.lam,
        "best_shift": best_shift_all,
        "w": best_w_all,
        "train_mse_all": best_mse_all,
        "train_rs_all": best_rs_all,
        "cv_mean_test_mse": best.mean_test_mse,
        "cv_mean_test_r": best.mean_test_r,
        "cv_mean_train_mse": best.mean_train_mse,
        "cv_mean_train_r": best.mean_train_r,
        "cv_details": best.per_fold,
    }
    return ordered_results, final_fit


with slow component

In [ ]:
# ===================== SWEEP OVER k (uses the helpers you pasted) =====================
import numpy as np
import pandas as pd

if __name__ == "__main__":
    EXPT = 20251113                     # ← change this to make a new experiment
    #EXPT = 7
    global RNG
    RNG   = np.random.default_rng(EXPT) # global fallback

    mode = "slow"
    
    rng_w0 = np.random.default_rng(EXPT + 1)   # new w0 each experiment
    #rng_w0 = np.random.default_rng(42)
    TRUE_LAG_MS = -350
    SHIFTS = candidate_lags_units(step_ms=10, max_ms=1500)
    W_SPREAD = 0.7

    #LAMBDAS = [0, 0.001, 0.01, 0.1, 1, 5, 10, 100, 1000, 100000]
    LAMBDAS = np.logspace(-3, 1.5, 15)
    TRIM_MS = 200

    # Fix GT weights ONCE for comparable cosine similarity across k
    w0 = 1.0 + W_SPREAD * rng_w0.normal(size=N_CH)
    #w0 = w0 / (np.linalg.norm(w0) or 1.0)

    # k in {4, 9, 12, 15, ..., 39}
    #ks = [4] + list(range(9, 20, 3))
    n_triplets = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 17, 20]

    rows = []
    for k in n_triplets:
        print(f"\n--- CV sweep at k={k} folds ---")
        # simulate 3*k trials (3 trials per test fold)
        sim_rng = np.random.default_rng(EXPT * 1_000 + k)  # unique per k & experiment
        #sim_rng = np.random.default_rng(10_000 + k)
        trials_og = simulate_component_dataset(
            n_triplets=k, triplet_lengths_s=(26.0, 18.0, 10.0),
            pupil_lag_ms=TRUE_LAG_MS, w_true_fixed=w0, rng=sim_rng,
            pupil_extra='ar1', pupil_extra_scale=0.5,
            lag_jitter_ms=abs(TRUE_LAG_MS)*0.25,
            eeg_weight_drift=True, eeg_noise_sd=0.2
        )

        # ----- CV to choose λ* -----
        splits = triplet_splits_from_trials(trials_og, seed=7)
        trials_comp = split_components_all_trials(trials_og, fs=FS, low_fc=0.25, band=(0.4, 0.7))

        if mode == "slow":
            trials = [{'X': tr['X_slow'], 'y': tr['y_slow'], 'w_true': tr['w_true'], 'c': tr['c']} for tr in trials_comp]
        elif mode == "fast":
            trials = [{'X': tr['X_fast'], 'y': tr['y_fast'], 'w_true': tr['w_true'], 'c': tr['c']} for tr in trials_comp]
        else:
            trials = trials_og

        ordered_cv, final_fit = cross_validate_l2_and_lag(
            trials=trials,
            lambdas=LAMBDAS,
            candidate_shifts=SHIFTS,
            trim_ms=TRIM_MS,
            k_folds=k, seed=7,
            splits=splits
        )

        order_kx = pd.DataFrame(ordered_cv)
        order_kx.drop(columns=['per_fold'], inplace=True)
        order_kx.to_csv(f"order_ridge_SLOW_k{k}.csv", index=False)

        best_cv = ordered_cv[0]  # highest mean test r (tie -> smaller λ)

        lam_star = final_fit["lambda"]
        best_shift_star = final_fit["best_shift"]
        lag_ms_star = int(best_shift_star) * 10
        w_star = final_fit["w"]
        train_mse_all_star = final_fit["train_mse_all"]
        mean_test_mse_star = final_fit["cv_mean_test_mse"]
        train_rs_all_star = final_fit["train_rs_all"]
        mean_test_rs_star = final_fit["cv_mean_test_r"]

        # cosine similarity at λ*
        num = float(np.dot(w0, w_star))
        den = (np.linalg.norm(w0) or 1.0) * (np.linalg.norm(w_star) or 1.0)
        cos_star = abs(num / den)

        # ===== λ = 0 (no reg) — DO NOT use CV to choose lag/weights =====
        # Choose lag+weights on ALL trials (search best lag over SHIFTS)
        best_mse0_all, rs_best_mse0, best_s0, w0_all = np.inf, np.nan, None, None
        for s in SHIFTS:
            w_s, mse_all, rs_all = fit_at_shift_lambda(trials, int(s), 0.0, TRIM_MS)
            if np.isfinite(mse_all) and mse_all < best_mse0_all:
                best_mse0_all, rs_best_mse0, best_s0, w0_all = mse_all, rs_all, int(s), w_s

        lag_ms_zero = int(best_s0) * 10
        train_mse_all_zero = float(best_mse0_all)
        train_rs_all_zero = float(rs_best_mse0)

        # For fairness, compute MEAN TEST corr for λ=0 by *evaluating* this fixed (lag,w) on the folds,
        # but not using any CV information to select them.
        rng = np.random.default_rng(7)
        splits = kfold_indices(len(trials), k, rng)
        test_mses_zero, test_rs_zero = [], []
        for tr_idx, te_idx in splits:
            te_set = [trials[i] for i in te_idx]
            mse_te0, r_te0 = evaluate_on_trials(te_set, best_s0, w0_all, TRIM_MS)
            test_mses_zero.append(mse_te0)
            test_rs_zero.append(r_te0)
        mean_test_mse_zero = float(np.nanmean(test_mses_zero))
        mean_test_rs_zero = float(np.nanmean(test_rs_zero))

        # cosine similarity at λ=0
        num0 = float(np.dot(w0, w0_all))
        den0 = (np.linalg.norm(w0) or 1.0) * (np.linalg.norm(w0_all) or 1.0)
        cos_zero = abs(num0 / den0)

        rows.append({
            "k": k,
            "lam_star": lam_star,
            "best_lag_ms_star": lag_ms_star,
            "train_mse_all_star": float(train_mse_all_star),
            "mean_test_mse_star": float(mean_test_mse_star),
            "train_r_all_star": float(train_rs_all_star),
            "mean_test_r_star": float(mean_test_rs_star),
            "w_star": w_star,
            "cos_star": float(cos_star),
            "best_lag_ms_lam0": lag_ms_zero,
            "train_mse_all_lam0": train_mse_all_zero,
            "mean_test_mse_lam0": mean_test_mse_zero,
            "train_r_all_lam0": train_rs_all_zero,
            "mean_test_lam0": mean_test_rs_zero,
            "w_lam0": w0_all,
            "cos_lam0": float(cos_zero),
        })
        dfk_temp = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
        dfk_temp.to_csv("dfk_ridge_TMP.csv", index=False)

    # ---- results table
    dfk = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
    dfk.to_csv("dfk_ridge.csv", index=False)

    print("\n=== CV sweep (by k) ===")
    with pd.option_context('display.max_rows', None, 'display.width', 140):
        print(dfk.to_string(index=False, float_format=lambda x: f"{x:.3f}"))



--- CV sweep at k=3 folds ---
[CV] Evaluating λ factor 0.001 out of 15


TypeError: '>=' not supported between instances of 'NoneType' and 'int'

In [8]:
import pandas as pd
dfk = pd.read_csv("dfk_ridge_TMP.csv")

print("\n=== CV sweep (by k) ===")
with pd.option_context('display.max_rows', None, 'display.width', 140):
    print(dfk.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# ---- plots vs k
import matplotlib.pyplot as plt

fig, axs = plt.subplots(3, 2, figsize=(11, 9), sharex=True)

axs[0,0].plot(dfk["k"], dfk["lam_star"], marker="o")
axs[0,0].set_title("Optimal λ vs k"); axs[0,0].set_ylabel("λ*"); axs[0,0].grid(alpha=.3)

axs[0,1].plot(dfk["k"], dfk["best_lag_ms_star"], marker="o", label="λ*")
axs[0,1].plot(dfk["k"], dfk["best_lag_ms_lam0"], marker="s", ls="--", label="λ=0")
axs[0,1].set_title("Best lag vs k"); axs[0,1].set_ylabel("Lag (ms)"); axs[0,1].legend(); axs[0,1].grid(alpha=.3)

axs[1,0].plot(dfk["k"], dfk["train_r_all_star"], marker="o", label="r λ*")
axs[1,0].plot(dfk["k"], dfk["train_r_all_lam0"], marker="s", ls="--", label="r λ=0")
axs[1, 0].plot(dfk["k"], dfk["train_mse_all_star"], marker="o", label="MSE λ*")
axs[1, 0].plot(dfk["k"], dfk["train_mse_all_lam0"], marker="s", ls="--", label="MSE λ=0")
axs[1,0].set_title("Train corr on ALL data"); axs[1,0].set_ylabel("corr"); axs[1,0].legend(); axs[1,0].grid(alpha=.3)

axs[1,1].plot(dfk["k"], dfk["mean_test_r_star"], marker="o", label="r λ*")
axs[1,1].plot(dfk["k"], dfk["mean_test_lam0"], marker="s", ls="--", label="r λ=0")
axs[1,1].plot(dfk["k"], dfk["mean_test_mse_star"], marker="o", label="MSE λ*")
axs[1,1].plot(dfk["k"], dfk["mean_test_mse_lam0"], marker="s", ls="--", label="MSE λ=0")
axs[1,1].set_title("Mean test corr vs k"); axs[1,1].set_ylabel("corr"); axs[1,1].legend(); axs[1,1].grid(alpha=.3)

axs[2,0].plot(dfk["k"], dfk["cos_star"], marker="o", label="λ*")
axs[2,0].plot(dfk["k"], dfk["cos_lam0"], marker="s", ls="--", label="λ=0")
axs[2,0].set_title("Cosine similarity vs k"); axs[2,0].set_ylabel("cosine"); axs[2,0].set_xlabel("k (folds; 3 trials per test)"); axs[2,0].legend(); axs[2,0].grid(alpha=.3)

axs[2,1].axis("off")
plt.tight_layout()
plt.show()



=== CV sweep (by k) ===
 k  lam_star  best_lag_ms_star  train_mse_all_star  mean_test_mse_star  train_r_all_star  mean_test_r_star                                                                                                                                                                                                            w_star  cos_star  best_lag_ms_lam0  train_mse_all_lam0  mean_test_mse_lam0  train_r_all_lam0  mean_test_lam0                                                                                                                                                                                                                                                                                    w_lam0  cos_lam0
 3     0.781                 0               0.170               0.192             0.913             0.902 [ 0.06950094  0.04551344 -0.00455896  0.0830288   0.1030812   0.02781512\n -0.04590467  0.01526814  0.05077649  0.01215141 -0.04982555 -0.02634075\n  0.09392

In [ ]:
alphas = np.logspace(-3, 1.5, 15)
print("Alphas:", alphas)

Alphas: [1.00000000e-03 2.09617999e-03 4.39397056e-03 9.21055318e-03
 1.93069773e-02 4.04708995e-02 8.48342898e-02 1.77827941e-01
 3.72759372e-01 7.81370738e-01 1.63789371e+00 3.43332002e+00
 7.19685673e+00 1.50859071e+01 3.16227766e+01]


In [7]:
%matplotlib qt

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
n_triplets = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 17, 20]


# Folder with your CSVs 
folder = r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg"

# The k values you mentioned
k_values = n_triplets
#k_values = [4]
# File name pattern: order_k{K}.csv
for k in k_values:
    path = os.path.join(folder, f"order_ridge_SLOW_k{k}.csv")
    if not os.path.isfile(path):
        print(f"[skip] File not found: {path}")
        continue

    df = pd.read_csv(path)

    # Ensure numeric & sort by lam
    for c in ["lam", "mean_train_mse", "mean_test_mse"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["lam", "mean_train_mse", "mean_test_mse"]).sort_values("lam")

    # Plot for this k
    plt.figure()
    plt.plot(df["lam"], df["mean_train_mse"], marker="o", label="mean_train_mse")
    plt.plot(df["lam"], df["mean_test_mse"], marker="s", label="mean_test_mse")
    plt.plot(df["lam"], df["mean_train_r"], marker="o", label="mean_train_r")
    plt.plot(df["lam"], df["mean_test_r"], marker="s", label="mean_test_r")
    plt.xlabel("lam")
    plt.ylabel("r")
    plt.title(f"Train/Test r vs lam — k={k}")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()

    # Save and optionally show
    out_path = os.path.join(folder, f"order_k{k}.png")
    # plt.savefig(out_path, dpi=200)
    plt.show()  # uncomment if you want to pop up the window
    #plt.close()

    print(f"[ok] Saved: {out_path}")


[ok] Saved: C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\order_k3.png
[ok] Saved: C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\order_k4.png
[ok] Saved: C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\order_k5.png
[ok] Saved: C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\order_k6.png
[ok] Saved: C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\order_k7.png
[ok] Saved: C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\order_k8.png
[ok] Saved: C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\simulated_check\simulation_ridge_reg\order_k9.png
[ok] Saved: C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\THESIS\si

In [ ]:
Sxx = (X_tr.T @ X_tr) / len(X_tr)
evals = np.linalg.eigvalsh(Sxx)
gains = evals / (evals + lam)
print(f"cond(Sxx)={evals.max()/max(evals.min(),1e-12):.1e}, "
      f"median_gain={np.median(gains):.3f}, min_gain={gains[0]:.3f}")


NameError: name 'X_tr' is not defined